##### Read bronze data and create Dataframe

In [1]:
customers_raw = spark.read.parquet("abfss://ECommerce_Project@onelake.dfs.fabric.microsoft.com/ECommerce_LakeHouse.Lakehouse/Files/bronze/customers.parquet")
orders_raw = spark.read.parquet("abfss://ECommerce_Project@onelake.dfs.fabric.microsoft.com/ECommerce_LakeHouse.Lakehouse/Files/bronze/orders.parquet")
payments_raw = spark.read.parquet("abfss://ECommerce_Project@onelake.dfs.fabric.microsoft.com/ECommerce_LakeHouse.Lakehouse/Files/bronze/payments.parquet")
support_tickets_raw = spark.read.parquet("abfss://ECommerce_Project@onelake.dfs.fabric.microsoft.com/ECommerce_LakeHouse.Lakehouse/Files/bronze/support_tickets.parquet")
web_activities_raw = spark.read.parquet("abfss://ECommerce_Project@onelake.dfs.fabric.microsoft.com/ECommerce_LakeHouse.Lakehouse/Files/bronze/web_activities.parquet")

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 3, Finished, Available, Finished, False)

In [2]:
display(customers_raw.limit(5))

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f5f60048-a4bb-4b9d-b03d-5fbe162ef55b)

In [3]:
display(orders_raw.limit(5))

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 231cb7cf-4862-401c-a020-a72a169ea97d)

In [4]:
display(payments_raw.limit(5))

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 96939105-ba64-4a60-a0cb-c945b7503715)

In [5]:
display(support_tickets_raw.limit(5))

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, c7a49194-02bc-4320-9d3a-1b273aabcf39)

In [6]:
display(web_activities_raw.limit(5))

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 42ed26ef-9011-4f0b-8f00-e8c51d82dc8d)

#### Create delta bronze table

In [7]:
# Save as Bronze Delta Tables
customers_raw.write.format("delta").mode("overwrite").saveAsTable("customers")
orders_raw.write.format("delta").mode("overwrite").saveAsTable("orders")
payments_raw.write.format("delta").mode("overwrite").saveAsTable("payments")
support_tickets_raw.write.format("delta").mode("overwrite").saveAsTable("support_tickets")
web_activities_raw.write.format("delta").mode("overwrite").saveAsTable("web_activities")

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 9, Finished, Available, Finished, False)

In [9]:
%%sql
select * from customers

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 11, Finished, Available, Finished, False)

<Spark SQL result set with 15 rows and 6 fields>

### cleaned the data - silver 

#### customer data -clean

In [8]:
display(customers_raw.limit(5))

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4e67f8a9-850e-4db4-ab33-fdafe68baabe)

In [10]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 12, Finished, Available, Finished, False)

In [12]:
customers_clean = (
    customers_raw
    .withColumn("email", lower(trim(col("EMAIL"))))
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn("gender", when(lower(col("gender")).isin("f", "female"), "Female")
                          .when(lower(col("gender")).isin("m", "male"), "Male")
                          .otherwise("Other"))
    .withColumn("dob", to_date(regexp_replace(col("dob"), "/", "-")))
    .withColumn("location", initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id", "email"])
)
display(customers_clean.limit(6))

customers_clean.write.format("delta").mode("overwrite").saveAsTable("silver_customers")

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 14, Finished, Available, Finished, False)

#### clean order table

In [13]:
%%sql 
select * from orders limit 5

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 15, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [14]:
orders = spark.table("orders")
orders_clean = (
    orders
    .withColumn("order_date", 
                when(col("order_date").rlike("^\d{4}/\d{2}/\d{2}$"), to_date(col("order_date"), "yyyy/MM/dd"))
                .when(col("order_date").rlike("^\d{2}-\d{2}-\d{4}$"), to_date(col("order_date"), "dd-MM-yyyy"))
                .when(col("order_date").rlike("^\d{8}$"), to_date(col("order_date"), "yyyyMMdd"))
                .otherwise(to_date(col("order_date"), "yyyy-MM-dd")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id", "order_date"])
    .dropDuplicates(["order_id"])
)
display(orders_clean.limit(5))
orders_clean.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f1545cf9-ef7b-4f63-a02d-2e8ae1e06be5)

#### Payments table - clean

In [15]:
%%sql 
select * from payments limit 5

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 17, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 6 fields>

In [16]:
payments = spark.table("payments")
payments_clean = (
    payments
    .withColumn("payment_date", to_date(regexp_replace(col("payment_date"), "/", "-")))
    .withColumn("payment_method", initcap(col("payment_method")))
    .replace({"creditcard": "Credit Card"}, subset=["payment_method"])
    .withColumn("payment_status", initcap(col("payment_status")))
    .withColumn("amount", col("amount").cast(DoubleType()))
    .withColumn("amount", when(col("amount") < 0, None).otherwise(col("amount")))
    .dropna(subset=["customer_id", "payment_date", "amount"])
)
display(payments_clean.limit(5))
payments_clean.write.format("delta").mode("overwrite").saveAsTable("silver_payments")


StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a53e44c1-15cd-436c-9578-eba5000c407a)

#### clean - support tables

In [17]:
%%sql
select * from support_tickets limit 5

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 19, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [19]:
support = spark.table("support_tickets")
support_clean = (
    support
    .withColumn("ticket_date", to_date(regexp_replace(col("ticket_date"), "/", "-")))
    .withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))
    .replace({"NA": None, "": None}, subset=["issue_type", "resolution_status"])
    .dropDuplicates(["ticket_id"])
    .dropna(subset=["customer_id", "ticket_date"])
)
display(support_clean.limit(5))

support_clean.write.format("delta").mode("overwrite").saveAsTable("silver_support_tickets")

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ff8cd283-a9f1-46ba-b1a6-74be1086acab)

#### - clean - web data 

In [20]:
%%sql 
select * from web_activities limit 5

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 22, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 5 fields>

In [21]:
web = spark.table("web_activities")
web_clean = (
    web
    .withColumn("session_time", to_date(regexp_replace(col("session_time"), "/", "-")))
    .withColumn("page_viewed", lower(col("page_viewed")))
    .withColumn("device_type", initcap(col("device_type")))
    .dropDuplicates(["session_id"])
    .dropna(subset=["customer_id", "session_time", "page_viewed"])
)
display(web_clean.limit(5))
web_clean.write.format("delta").mode("overwrite").saveAsTable("silver_web_activities")


StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 23, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1e1f5658-093b-4f10-b339-2b3fc8d761dd)

## GOLD TABLES - AGGRGATE TABLES

In [22]:
cust = spark.table("silver_customers").alias("c")
orders = spark.table("silver_orders").alias("o")
payments = spark.table("silver_payments").alias("p")
support = spark.table("silver_support_tickets").alias("s")
web = spark.table("silver_web_activities").alias("w")

customer360 = (
    cust
    .join(orders, "customer_id", "left")
    .join(payments, "customer_id", "left")
    .join(support, "customer_id", "left")
    .join(web, "customer_id", "left")
    .select(
        col("c.customer_id"),
        col("c.name"),
        col("c.email"),
        col("c.gender"),
        col("c.dob"),
        col("c.location"),

        col("o.order_id"),
        col("o.order_date"),
        col("o.amount").alias("order_amount"),
        col("o.status").alias("order_status"),

        col("p.payment_method"),
        col("p.payment_status"),
        col("p.amount").alias("payment_amount"),

        col("s.ticket_id"),
        col("s.issue_type"),
        col("s.ticket_date"),
        col("s.resolution_status"),

        col("w.page_viewed"),
        col("w.device_type"),
        col("w.session_time")
    )
)
display(customer360.limit(10))

customer360.write.format("delta").mode("overwrite").saveAsTable("gold_customer360")

StatementMeta(, 392bf1d0-7980-4e7d-b89d-6a2bb126509b, 24, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e5d5c2eb-bded-48b6-adc9-701c81ae6861)